# 🚀 GARRO: Research-Grade DRL Routing Orchestrator for Data Centers
## ⚡ Kaggle Cloud Training Pipeline: Dual Tesla T4 GPUs (31.2 GB VRAM) | 4 CPU Cores | 30 GB RAM

This notebook trains **GARRO** (Graph Attention Routing with Reinforcement Learning) on a **Fat-Tree ($k=4$) Data Center Network (DCN)** fabric under production traffic conditions.

### 🌟 Key Innovations
1. **Path-Centric Attention (RouteNet-Fermi Paradigm)**: Replaces global MLP pooling with explicit constituent link and bottleneck aggregation ($h_{\text{bottleneck}} = \max_{e \in P} h_e$) to eliminate ECMP collisions.
2. **Realistic DCN Traffic Dynamics**: Diurnal dual-wave time-of-day modulation, Pareto elephant flows ($\alpha=1.15$), incast microbursts, bisection stride stress, and AllReduce collective exchange.
3. **Multi-GPU Parallelism (DDP)**: Distributed Data Parallel training across Dual Tesla T4 GPUs utilizing NCCL ring-allreduce.
4. **Vectorized Multi-Environment Rollout**: $N=32$ parallel environments ($16$ envs/GPU) to saturate Turing Tensor Cores and bypass the 4-CPU bottleneck.
5. **3-Stage Curriculum Learning**: Warmup Exploration $\to$ Heavy Bimodal Stress $\to$ Chaos & Incast Collective Bursts.

## 1. Hardware & Accelerator Verification
Check GPU availability, VRAM allocation, CPU core count, and system memory.

In [ ]:
!nvidia-smi

import os
import multiprocessing
import psutil
import torch

n_gpus = torch.cuda.device_count()
n_cpus = multiprocessing.cpu_count()
ram_gb = psutil.virtual_memory().total / (1024 ** 3)

print(f"\n{'='*55}")
print(f"  Kaggle Hardware Verification Report")
print(f"{'='*55}")
print(f"  GPUs Detected     : {n_gpus}")
for i in range(n_gpus):
    props = torch.cuda.get_device_properties(i)
    print(f"    GPU {i}: {props.name} | VRAM: {props.total_memory / (1024**3):.2f} GB")
print(f"  CPU Cores         : {n_cpus}")
print(f"  System RAM        : {ram_gb:.2f} GB")
print(f"  CUDA Available    : {torch.cuda.is_available()}")
if n_gpus >= 2:
    print("  STATUS            : READY FOR DUAL T4 DDP ACCELERATION ✓")
else:
    print("  STATUS            : WARNING: Set Accelerator to 'GPU T4 x2' in Kaggle Settings")
print(f"{'='*55}")

## 2. Dependencies & Environment Setup
Install PyTorch Geometric, NetworkX, SciPy, and other required packages.

In [ ]:
!pip install --quiet torch-geometric networkx scipy pyyaml matplotlib tqdm psutil tabulate
print("All core dependencies verified and installed successfully! ✓")

## 3. Verify GARRO Modules & Traffic Engine

In [ ]:
from digital_twin.topologies import * if False else None
from topologies.fat_tree import get_fat_tree
from digital_twin.traffic_generator import create_traffic_generator
from model.graph_transformer import PathAttentionActorCritic

G = get_fat_tree(k=4)
print(f"Fat-Tree Fabric: {G.number_of_nodes()} switches, {G.number_of_edges()} bidirectional links")

gen = create_traffic_generator(G, mode="datacenter", base_rate=100.0, pareto_prob=0.25)
print("Traffic Generator mode: Data Center (Pareto, Incast, Diurnal wave active)")

flow = gen.sample_flow_pair()
print(f"Sampled Flow: src={flow.src}, dst={flow.dst}, volume={flow.volume:.2f} Mbps, elephant={flow.is_elephant}")
print("Path-Centric Engine is ready! ✓")

## 4. Launch Multi-GPU DDP 50,000-Episode Training
Run `torchrun --nproc_per_node=2` to train GARRO across both Tesla T4 GPUs in parallel.

### Training Parameters:
- **Total Episodes**: 50,000
- **Parallel Environments**: 16 per GPU (32 parallel environments total)
- **Mixed Precision**: FP16 (Turing Tensor Cores)
- **Curriculum Stages**:
  - *Stage 1 (0–10k eps)*: Low load discovery ($\alpha=1.5$, uniform traffic)
  - *Stage 2 (10k–35k eps)*: Heavy bimodal stress (Pareto elephant flows, stride bisection stress)
  - *Stage 3 (35k–50k eps)*: Chaos diurnal incast bursts & AllReduce collective communication
- **Checkpoint Interval**: Every 2,500 episodes with automated resume on session restart

In [ ]:
!torchrun --nproc_per_node=2 train_kaggle.py \
    --topology fat_tree \
    --episodes 50000 \
    --num-envs 16 \
    --checkpoint-every 2500 \
    --checkpoint-dir checkpoints

## 5. Live Telemetry & Training Metrics Visualization
Plot moving-average rewards, policy loss, and critic value estimation.

In [ ]:
from IPython.display import Image, display
import os

curve_img = "checkpoints/training_curve_fat_tree.png"
if os.path.exists(curve_img):
    display(Image(curve_img))
else:
    print("Training curve image not yet generated. Check after first checkpoint.")

## 6. Comprehensive DCN Benchmark Evaluation
Evaluate the trained GARRO checkpoint against **ECMP (Equal-Cost Multi-Path)**, **OSPF (Dijkstra)**, and **K-Shortest Paths (KSP)** under synchronized incast microbursts and elephant flow collisions.

In [ ]:
!python evaluate.py \
    --topology fat_tree \
    --checkpoint checkpoints/garro_fat_tree_final.pt \
    --episodes 500 \
    --compare-all

## 7. Package Checkpoints & Artifacts for Download
Archive trained checkpoints, telemetry logs, and evaluation figures into a single downloadable zip file.

In [ ]:
!tar -czvf garro_kaggle_trained_artifacts.tar.gz checkpoints/ *.png
print("Artifacts compressed successfully! Ready to download via Kaggle output file browser.")